# 08 - Final primary experiment

Restart-safe orchestration for `main-final-primary`. The saved notebook never authorizes inference; execution copies may enable only the local stage gates.

## 1. Provenance and safe controls

In [ ]:
from pathlib import Path

from rubric_order.api import find_project_root, load_experiment_config

PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "main_final_primary.yaml"
DATA_PATH = PROJECT_ROOT / "HelpSteer2"
RUN_ID = "main-final-primary"
RUNTIME_OVERRIDES = {}
EXECUTE_MODEL_CALLS = False
MAX_CALLS = 0
assert EXECUTE_MODEL_CALLS is False and MAX_CALLS == 0
config = load_experiment_config(CONFIG_PATH, data_path=DATA_PATH, overrides=RUNTIME_OVERRIDES)
target = Path(config["output_root"]) / RUN_ID
if not target.is_absolute():
    target = PROJECT_ROOT / target
EXECUTED_NOTEBOOK_PATH = target / "notebooks" / "08_final_main_experiment.executed.ipynb"
provenance = {
    "config": str(CONFIG_PATH),
    "run_id": RUN_ID,
    "target_exists": target.exists(),
    "global_model_calls_enabled": EXECUTE_MODEL_CALLS,
}

## 2. Approved judge

In [ ]:
from rubric_order.downstream import verify_approved_judge

assert EXECUTE_MODEL_CALLS is False and MAX_CALLS == 0
approved_judge = verify_approved_judge(config)
assert approved_judge["model"] == "qwen3:14b"
assert (
    approved_judge["prompt_sha256"]
    == "75dd61241b143deabe1c9e9435f897c17874ea4cc7ed0b513c43cd35687cda7c"
)

## 3. Exclusions

In [ ]:
assert EXECUTE_MODEL_CALLS is False and MAX_CALLS == 0
required_exclusions = {
    "runs/pilot-001/manifest.json",
    "runs/judge-validation-v2/manifest.json",
    "runs/pilot-002/manifest.json",
}
configured_exclusions = {row["path"].replace("\\", "/") for row in config["exclude_manifests"]}
assert configured_exclusions == required_exclusions
exclusion_status = {path: (PROJECT_ROOT / path).is_file() for path in sorted(required_exclusions)}
assert all(exclusion_status.values())

## 4. Dataset and sampling (read-only)

In [ ]:
from rubric_order.api import validate_dataset

assert EXECUTE_MODEL_CALLS is False and MAX_CALLS == 0
dataset_preview = validate_dataset(config, preview_rows=2)
sampling_contract = {
    "blocks": config["sampling"]["calibration_blocks"],
    "heldout_prompts": config["sampling"]["heldout_prompts"],
    "responses_per_prompt": config["sampling"]["responses_per_prompt"],
    "materialized": target.exists(),
}

## 5. Exact estimator

In [ ]:
from rubric_order.final_primary import estimate_final_primary

assert EXECUTE_MODEL_CALLS is False and MAX_CALLS == 0
estimate = estimate_final_primary(config)
assert estimate["physical_calls"] == 5156
assert estimate["breakdown"]["rubric_stage_calls"] == 506
assert estimate["breakdown"]["judge_stage_calls"] == 4650
assert config["execution_safety"]["hard_total_call_ceiling"] == 5200
assert estimate["constrained_calls"] == 0

## 6. Updater dry-run

In [ ]:
from rubric_order.api import plan_rubric_generation

assert EXECUTE_MODEL_CALLS is False and MAX_CALLS == 0
updater_dry_run = (
    plan_rubric_generation(config, RUN_ID)
    if target.exists()
    else {"status": "blocked_until_run_is_prepared", "executed": False}
)

## 7. Updater canary

In [ ]:
from rubric_order.api import build_rubrics, execution_safety_status

RUN_UPDATER_CANARY = False
UPDATER_CANARY_METHODS = ["batch"]
UPDATER_CANARY_BLOCKS = [0]
UPDATER_CANARY_ORDERS = ["ascending"]
UPDATER_CANARY_REPLICATES = [0]
UPDATER_CANARY_MAX_CALLS = 1
assert EXECUTE_MODEL_CALLS is False and MAX_CALLS == 0
updater_canary_selection = {
    "methods": UPDATER_CANARY_METHODS,
    "blocks": UPDATER_CANARY_BLOCKS,
    "orders": UPDATER_CANARY_ORDERS,
    "replicates": UPDATER_CANARY_REPLICATES,
}
if RUN_UPDATER_CANARY:
    if not target.exists():
        raise RuntimeError("prepare main-final-primary before updater canary")
    updater_canary_preflight = plan_rubric_generation(config, RUN_ID, **updater_canary_selection)
    assert updater_canary_preflight["planned_physical_updater_calls"] == 1
    assert updater_canary_preflight["execution_fresh_calls"] in {0, 1}
    updater_canary = build_rubrics(
        config,
        RUN_ID,
        execute_model_calls=RUN_UPDATER_CANARY,
        max_calls=UPDATER_CANARY_MAX_CALLS,
        **updater_canary_selection,
    )
    assert updater_canary["actual_calls"] == updater_canary_preflight["execution_fresh_calls"]
    assert updater_canary["final_phase_plan"]["execution_fresh_calls"] == 0
    assert execution_safety_status(config, RUN_ID)["safe"]
else:
    updater_canary = {"status": "disabled", "executed": False, "actual_calls": 0}

## 8. Updater full resume

In [ ]:
RUN_UPDATER_FULL = False
UPDATER_FULL_MAX_CALLS = 519
assert EXECUTE_MODEL_CALLS is False and MAX_CALLS == 0
if RUN_UPDATER_FULL:
    updater_full_preflight = plan_rubric_generation(config, RUN_ID)
    assert updater_full_preflight["planned_physical_updater_calls"] == 506
    assert updater_full_preflight["execution_fresh_calls"] <= 505
    updater_full = build_rubrics(
        config, RUN_ID, execute_model_calls=RUN_UPDATER_FULL, max_calls=UPDATER_FULL_MAX_CALLS
    )
    assert updater_full["final_phase_plan"]["execution_fresh_calls"] == 0
    assert updater_full["final_phase_plan"]["deferred_requests"] == 0
    assert updater_full["final_phase_plan"]["invalid_requests"] == 0
    assert execution_safety_status(config, RUN_ID)["safe"]
else:
    updater_full = {"status": "disabled", "executed": False, "actual_calls": 0}

## 9. Optional blinded rubric export

In [ ]:
from rubric_order.drift_annotations import export_blinded_drift, plan_blinded_drift_export

RUN_OPTIONAL_DRIFT_EXPORT = False
assert EXECUTE_MODEL_CALLS is False and MAX_CALLS == 0
rubric_index = target / "artifacts" / "rubrics.jsonl"
if rubric_index.exists():
    drift_plan = plan_blinded_drift_export(target)
    drift_export = (
        export_blinded_drift(target, target / "artifacts" / "drift_annotations" / "export")
        if RUN_OPTIONAL_DRIFT_EXPORT
        else {"status": "optional_not_run", "blocking": False}
    )
else:
    drift_plan = {"status": "blocked_until_rubrics_exist", "blocking": False}
    drift_export = {"status": "not_run", "blocking": False}

## 10. Main annotation policy (non-blocking)

In [ ]:
assert EXECUTE_MODEL_CALLS is False and MAX_CALLS == 0
annotation_policy = config["drift_annotations"]
assert annotation_policy["required_before_judging"] is False
assert annotation_policy["optional_posthoc_inspection_only"] is True
assert annotation_policy["manual_semantic_drift_rate_estimated_in_main"] is False
assert annotation_policy["lexical_flags_are_ground_truth"] is False

## 11. Judge dry-run

In [ ]:
from rubric_order.api import plan_judging

assert EXECUTE_MODEL_CALLS is False and MAX_CALLS == 0
judge_dry_run = (
    plan_judging(config, RUN_ID)
    if rubric_index.exists()
    else {"status": "blocked_until_rubrics_exist", "executed": False}
)
if rubric_index.exists():
    assert judge_dry_run["annotation_freeze"]["required"] is False
    assert judge_dry_run["expected_physical_requests"] == 4650

## 12. Judge canary

In [ ]:
from rubric_order.api import judge_responses

RUN_JUDGE_CANARY = False
JUDGE_CANARY_METHODS = ["static"]
JUDGE_CANARY_BLOCKS = [0]
JUDGE_CANARY_ORDERS = ["ascending"]
JUDGE_CANARY_REPLICATES = [0]
JUDGE_CANARY_MAX_CALLS = 1
JUDGE_CANARY_ALLOW_PARTIAL = True
assert EXECUTE_MODEL_CALLS is False and MAX_CALLS == 0
judge_canary_selection = {
    "methods": JUDGE_CANARY_METHODS,
    "blocks": JUDGE_CANARY_BLOCKS,
    "orders": JUDGE_CANARY_ORDERS,
    "replicates": JUDGE_CANARY_REPLICATES,
}
if RUN_JUDGE_CANARY:
    judge_canary_preflight = plan_judging(config, RUN_ID, **judge_canary_selection)
    assert judge_canary_preflight["ready"]
    assert judge_canary_preflight["physical_requests"] == 50
    if judge_canary_preflight["cache_hits"] >= 1:
        judge_canary = {"status": "reused", "executed": False, "actual_calls": 0}
    else:
        judge_canary = judge_responses(
            config,
            RUN_ID,
            execute_model_calls=RUN_JUDGE_CANARY,
            max_calls=JUDGE_CANARY_MAX_CALLS,
            allow_partial_execution=JUDGE_CANARY_ALLOW_PARTIAL,
            **judge_canary_selection,
        )
        assert judge_canary["actual_calls"] == 1
        assert judge_canary["judgment_summary"]["completed"] == 1
    assert execution_safety_status(config, RUN_ID)["safe"]
else:
    judge_canary = {"status": "disabled", "executed": False, "actual_calls": 0}

## 13. Judge full resume

In [ ]:
RUN_JUDGE_FULL = False
JUDGE_FULL_MAX_CALLS = 4679
assert EXECUTE_MODEL_CALLS is False and MAX_CALLS == 0
if RUN_JUDGE_FULL:
    judge_full_preflight = plan_judging(config, RUN_ID)
    assert judge_full_preflight["ready"]
    assert judge_full_preflight["physical_requests"] == 4650
    assert judge_full_preflight["fresh_requests"] <= 4649
    judge_full = judge_responses(
        config, RUN_ID, execute_model_calls=RUN_JUDGE_FULL, max_calls=JUDGE_FULL_MAX_CALLS
    )
    assert judge_full["status"] == "complete"
    assert judge_full["final_phase_plan"]["fresh_requests"] == 0
    assert judge_full["final_phase_plan"]["invalid_requests"] == 0
    assert execution_safety_status(config, RUN_ID)["safe"]
else:
    judge_full = {"status": "disabled", "executed": False, "actual_calls": 0}

## 14. Final analysis

In [ ]:
from rubric_order.api import analyze_experiment

RUN_FINAL_ANALYSIS = False
assert EXECUTE_MODEL_CALLS is False and MAX_CALLS == 0
predictions_path = target / "artifacts" / "predictions.jsonl"
if RUN_FINAL_ANALYSIS:
    final_analysis = analyze_experiment(config, RUN_ID, allow_partial_results=False)
    assert final_analysis["completeness"]["complete"]
else:
    final_analysis = {
        "status": "disabled",
        "ready": predictions_path.exists(),
        "primary_question": config["paper_design"]["primary_question"],
    }

## 15. Final readiness

In [ ]:
from rubric_order.final_primary import final_main_readiness

RUN_FINAL_READINESS = False
assert EXECUTE_MODEL_CALLS is False and MAX_CALLS == 0
if RUN_FINAL_READINESS:
    final_readiness = final_main_readiness(config, RUN_ID, executed_notebook=EXECUTED_NOTEBOOK_PATH)
    assert final_readiness["verdict"] == "MAIN_EXPERIMENT_COMPLETE"
else:
    final_readiness = {"status": "disabled", "verdict": None}
assert estimate["physical_calls"] == 5156

## Exact cell order

Run top-to-bottom: provenance, approved judge, exclusions, dataset/sampling, estimator, updater dry-run, updater canary, updater full resume, optional rubric export, non-blocking annotation policy, judge dry-run, judge canary, judge full resume, analysis, readiness. Keep the global `EXECUTE_MODEL_CALLS=False` and `MAX_CALLS=0`; authorize only the four local inference gates in an execution copy.